[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/01_baseline_evaluation.ipynb)

# Step 1 — Baseline Evaluation

Build a held-out **policy-document test set** and measure how a small instruction-tuned model performs before synthetic-data alignment.

## Learning objectives
- Ingest two finance policy documents (policy-dense + scope-boundary)
- Split paragraphs into **test** vs **train** sets
- Generate hard test Q&A with a teacher LLM
- Run baseline inference and LLM-as-judge scoring by failure mode

flowchart LR
    A[Pick domain & tasks] --> B[Build test set]
    B --> C[Run baseline model]
    C --> D[LLM judge scores answers]
    D --> E[Generate synthetic data]
    E --> F[Fine-tune model]
    F --> G[Re-run same test]
    G --> H[Compare scores]


## Setup

In [2]:
from pathlib import Path
import os
from aieng.syn_data.text import (
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SCORES_PATH,
    DEFAULT_TEST_PARAS_PER_DOC,
    PARAGRAPHS_PATH,
    TEST_SET_PATH,
    ParagraphSplit,
    QASample,
    build_paragraph_splits,
    create_judge_client,
    create_small_model_client,
    create_teacher_client,
    generate_test_qa_batch,
    list_domain_documents,
    run_inference,
    save_baseline_results,
    save_typed_jsonl,
    score_predictions,
    use_repo_root,
)
from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table
from rich.progress import track
from rich.panel import Panel


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_dotenv()
# ROOT = use_repo_root(Path("."))
console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [3]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,  # override any earlier basicConfig from other libs
)

In [4]:
# TODO: remove this before merging into main

%load_ext autoreload
%autoreload 2

## 1. Load finance policy documents

We use two document archetypes:
- **Policy-dense** (CFPB credit card agreement) → format + vocabulary + multi-constraint
- **Scope-boundary** (SEC investor bulletin) → refusal calibration

Two document types = two different skills the small model needs to learn.

Policy-dense (CFPB credit card agreement)

Lots of rules, numbers, fees, defined terms
Tests: format compliance (answer as JSON/table), domain vocabulary (APR, grace period), multi-constraint questions (“what’s the fee and when is it charged?”)
Scope-boundary (SEC investor bulletin)

Explains what the document covers — and what it doesn’t
Tests: refusal calibration — answer in-scope questions, politely refuse out-of-scope ones (e.g. “Should I buy this stock?”)

So the bootcamp uses two archetypes to build a test set and training data that stress different weaknesses — closer to real deployments where models handle both “answer precisely from policy” and “know their limits.”

In code, ``failure_modes_for_paragraph()`` maps each role to the failure modes it’s meant to target.

In [4]:
specs = list_domain_documents("finance")
specs

[DocumentSpec(doc_id='cfpb_credit_card_agreement', title='CFPB Sample Credit Card Agreement', role=<DocumentRole.POLICY_DENSE: 'policy_dense'>, domain='finance', source_url='https://files.consumerfinance.gov/f/documents/201401_cfpb_credit-card-agreement_english.pdf', local_path='implementations/qa_text_generation/data/documents/cfpb_credit_card_agreement.txt'),
 DocumentSpec(doc_id='sec_investor_bulletin', title='SEC Investor Bulletin', role=<DocumentRole.SCOPE_BOUNDARY: 'scope_boundary'>, domain='finance', source_url='https://www.sec.gov/files/ib_fraud.pdf', local_path='implementations/qa_text_generation/data/documents/sec_investor_bulletin.txt')]

## 2. Chunk into paragraphs and hold out test paragraphs

Randomly sample a few paragraphs per document for evaluation. **Never** use these paragraphs in Step 4 training.

In [5]:
paragraphs = build_paragraph_splits(
    "finance",
    n_test_per_doc=DEFAULT_TEST_PARAS_PER_DOC,
    seed=42,
)
test_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TEST]
train_paragraphs = [p for p in paragraphs if p.split == ParagraphSplit.TRAIN]

console.print(f"[bold green]Total paragraphs:[/bold green] [cyan]{len(paragraphs)}[/cyan]")
console.print(f"[bold green]Test holdout:[/bold green] [cyan]{len(test_paragraphs)}[/cyan] | [bold green]Train reserve:[/bold green] [cyan]{len(train_paragraphs)}[/cyan]")

save_typed_jsonl(
    PARAGRAPHS_PATH,
    paragraphs,
    to_dict=lambda paragraph: paragraph.to_dict(),
)
PARAGRAPHS_PATH

Total paragraphs: 56

Test holdout: 20 | Train reserve: 36

PosixPath('implementations/qa_text_generation/data/paragraphs.jsonl')

## 3. Generate hard test Q&A with the teacher model

Target the four small-model failure modes:
1. Format non-compliance
2. Domain vocabulary drift
3. Refusal vs engagement calibration
4. Multi-constraint collapse

In [8]:
teacher = create_teacher_client()

console.print(f"[bold magenta]Teacher LLM Base URL:[/bold magenta] [cyan]{teacher.settings.base_url}[/cyan]")

test_samples = generate_test_qa_batch(
    teacher,
    test_paragraphs,
    questions_per_para=3,
)
console.print(f"[bold green]Generated:[/bold green] [cyan]{len(test_samples)}[/cyan] test Q&A items")

save_typed_jsonl(
    TEST_SET_PATH,
    test_samples,
    to_dict=QASample.to_dict,
)

Teacher LLM Base URL: https://proxy.vectorinstitute.ai/v1

2026-06-23 21:16:39,115 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443


2026-06-23 21:16:41,783 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-06-23 21:16:41,785 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "topics": [
    "Finance charge grace periods for new purchases",
    "Accrual timeline for cash advance finance charges",
    "Calculation method for average daily balance of purchases",
    "Calculation method for average daily balance of cash advances",
    "Treatment of balance transfers regarding finance charges"
  ]
}
*********** End of JSON payload ***********
2026-06-23 21:16:41,787 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443
2026-06-23 21:16:43,393 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-06-23 21:16:43,395 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
  "question": "A

Generated: 60 test Q&A items

PosixPath('implementations/qa_text_generation/data/test/test_set.jsonl')

Altrnatively, you may already saved the generated tests. So you can continue with reading them without generation:

In [5]:
from aieng.syn_data.text.io import load_typed_jsonl


test_samples = load_typed_jsonl(TEST_SET_PATH, QASample.from_dict)

In [6]:
def show_qa_samples(samples):
    table = Table(title="Test Q&A Samples", show_lines=True)
    table.add_column("ID", style="cyan", no_wrap=True)
    table.add_column("Question", style="magenta")
    table.add_column("Answer", style="green")
    table.add_column("Failure Mode", style="yellow")
    table.add_column("Role", style="blue")

    for sample in samples:
        table.add_row(
            sample.id,
            sample.question[:60] + "..." if len(sample.question) > 60 else sample.question,
            sample.gold_answer[:60] + "..." if len(sample.gold_answer) > 60 else sample.gold_answer,
            str(sample.failure_mode.value) if hasattr(sample.failure_mode, 'value') else str(sample.failure_mode),
            str(sample.role.value) if hasattr(sample.role, 'value') else str(sample.role),
        )

    console.print(table)


In [7]:

show_qa_samples(test_samples[25:27])

                                          Test Q&A Samples                                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃                                          ┃             ┃             ┃ Failure     ┃             ┃
┃ ID                                       ┃ Question    ┃ Answer      ┃ Mode        ┃ Role        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ test-cfpb_credit_card_agreement::p0029-1 │ Under the   │ If the      │ domain_voc… │ policy_den… │
│                                          │ Vermont Law │ borrower    │             │             │
│                                          │ Notice to   │ does not    │             │             │
│                                          │ Co-signer,  │ pay, the    │             │             │
│                                          │ what        │ lender has  │             │             │
│                                          │ specific    │ a legal     │             │             │
│                                          │ leg...      │ right t...  │             │             │
├──────────────────────────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┤
│ test-cfpb_credit_card_agreement::p0029-2 │ Under       │ Under       │ multi_cons… │ policy_den… │
│                                          │ Vermont     │ Vermont     │             │             │
│                                          │ law, if a   │ law, the    │             │             │
│                                          │ co-signer   │ co-signer   │             │             │
│                                          │ signs a     │ is equally  │             │             │
│                                          │ loan note,  │ liable for  │             │             │
│                                          │ what is...  │ repay...    │             │             │
└──────────────────────────────────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

## 4. Baseline inference with the small model

Plug in your small model client here (local GGUF, Ollama, or HF 4-bit model).

In [9]:
small_model = create_small_model_client()

predictions = run_inference(small_model, test_samples)
console.print(f"[bold green]Collected:[/bold green] [cyan]{len(predictions)}[/cyan] baseline predictions")
from rich.panel import Panel
from rich.syntax import Syntax

sample = predictions[0]
sample_dict = sample.to_dict() if hasattr(sample, "to_dict") else dict(sample)
pretty_json = Syntax.from_json(sample_dict, indent=2) if hasattr(Syntax, "from_json") else None

if pretty_json is not None:
    console.print(Panel(pretty_json, title="First Prediction"))
else:
    # Fallback if Syntax.from_json is not available
    import json
    formatted = json.dumps(sample_dict, indent=2)
    console.print(Panel(formatted, title="First Prediction"))

2026-06-24 23:54:11,653 INFO aieng.syn_data.text.small_model: Creating small model client for model: qwen2.5:3b-instruct
2026-06-24 23:54:11,656 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434


2026-06-24 23:55:29,578 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 631
2026-06-24 23:55:29,581 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-06-24 23:55:31,107 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 829
2026-06-24 23:55:31,109 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-06-24 23:55:35,226 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 1876
2026-06-24 23:55:35,228 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-06-24 23:55:36,161 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/completions HTTP/1.1" 200 595
2026-06-24 23:55:36,163 DEBUG urllib3.connectionpool: Starting new HTTP connection (1): 127.0.0.1:11434
2026-06-24 23:55:37,252 DEBUG urllib3.connectionpool: http://127.0.0.1:11434 "POST /v1/chat/com

Collected: 60 baseline predictions

╭──────────────────────────────────────── First Prediction ────────────────────────────────────────╮
│ {                                                                                                │
│   "id": "test-cfpb_credit_card_agreement::p0005-0",                                              │
│   "question": "According to the policy, what are the two specific conditions under which new     │
│ purchases posted to an account during a billing cycle will not incur a finance charge for that   │
│ cycle? Please format your response as a numbered list containing exactly two items, citing the   │
│ conditions verbatim or near-verbatim from the text.",                                            │
│   "gold_answer": "1. If you had a zero or credit balance at the beginning of that billing        │
│ cycle.\n2. If you paid the entire new balance on the previous cycle's billing statement by the   │
│ payment due date of that statement.",                                                            │
│   "model_answer": "1. The new purchases posted to your account during a billing cycle will not   │
│ incur a finance charge if you had a zero or credit balance at the beginning of that billing      │
│ cycle.\n2. The new purchases also will not incur a finance charge if you paid the entire new     │
│ balance on the previous cycle's billing statement by its payment due date.",                     │
│   "failure_mode": "format_non_compliance",                                                       │
│   "doc_id": "cfpb_credit_card_agreement",                                                        │
│   "para_id": "cfpb_credit_card_agreement::p0005"                                                 │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

## 5. LLM-as-judge baseline scores

In [10]:
judge = create_judge_client()

baseline_scores = score_predictions(judge, test_samples, predictions)
baseline_summary = save_baseline_results(
    predictions,
    baseline_scores,
    test_samples,
    predictions_path=BASELINE_PREDICTIONS_PATH,
    scores_path=BASELINE_SCORES_PATH,
)

2026-06-24 23:58:42,089 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443
2026-06-24 23:58:43,895 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-06-24 23:58:43,898 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The model perfectly followed all instructions, including formatting the response as a numbered list with exactly two items and citing the conditions near-verbatim from the reference."
text."
."
}
*********** End of JSON payload ***********
2026-06-24 23:58:43,903 DEBUG urllib3.connectionpool: Starting new HTTPS connection (1): proxy.vectorinstitute.ai:443
2026-06-24 23:58:45,256 DEBUG urllib3.connectionpool: https://proxy.vectorinstitute.ai:443 "POST /v1/chat/completions HTTP/1.1" 200 None
2026-06-24 23:58:

### Baseline score summary

In [11]:

# Create a table to display the baseline_summary
table = Table(title="Baseline Summary", highlight=True)

# Define the columns based on the baseline_summary structure
table.add_column("Failure Mode", style="bold cyan")
table.add_column("Correctness", justify="right", style="green")
table.add_column("Coherence", justify="right", style="green")
table.add_column("Instruction Following", justify="right", style="green")
table.add_column("Factual Plausibility", justify="right", style="green")
table.add_column("Average", justify="right", style="bold yellow")

# Add the "overall" scores as the first row
overall = baseline_summary.get("overall", {})
table.add_row(
    "[b]Overall[/b]",
    f"{overall.get('correctness', 0):.2f}",
    f"{overall.get('coherence', 0):.2f}",
    f"{overall.get('instruction_following', 0):.2f}",
    f"{overall.get('factual_plausibility', 0):.2f}",
    f"{overall.get('average', 0):.2f}",
)

# Add a row for each failure mode
by_failure = baseline_summary.get("by_failure_mode", {})
for mode, scores in by_failure.items():
    table.add_row(
        mode,
        f"{scores.get('correctness', 0):.2f}",
        f"{scores.get('coherence', 0):.2f}",
        f"{scores.get('instruction_following', 0):.2f}",
        f"{scores.get('factual_plausibility', 0):.2f}",
        f"{scores.get('average', 0):.2f}",
    )

console.print(table)

                                          Baseline Summary                                          
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃                    ┃             ┃           ┃        Instruction ┃            Factual ┃         ┃
┃ Failure Mode       ┃ Correctness ┃ Coherence ┃          Following ┃       Plausibility ┃ Average ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ Overall            │        4.47 │      4.95 │               4.85 │               4.65 │    4.73 │
│ format_non_compli… │        4.55 │      5.00 │               4.75 │               4.70 │    4.75 │
│ domain_vocabulary… │        4.30 │      5.00 │               5.00 │               4.50 │    4.70 │
│ multi_constraint_… │        4.00 │      4.70 │               5.00 │               4.20 │    4.47 │
│ refusal_calibrati… │        4.70 │      5.00 │               4.80 │               4.90 │    4.85 │
└────────────────────┴─────────────┴───────────┴────────────────────┴────────────────────┴─────────┘